<a href="https://www.kaggle.com/code/raihankh/caramut-model-train?scriptVersionId=251506985" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Installasi Libraries

In [1]:
# Hapus semua versi lama untuk memastikan kebersihan lingkungan
!pip uninstall -y transformers datasets evaluate accelerate rouge_score

# Instal satu set versi spesifik yang dijamin kompatibel
!pip install transformers==4.28.1 datasets==2.12.0 evaluate==0.4.0 accelerate==0.19.0 rouge_score

# Instal library tambahan untuk widgets
!pip install --upgrade ipywidgets

Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Successfully uninstalled datasets-3.6.0
Found existing installation: accelerate 1.5.2
Uninstalling accelerate-1.5.2:
  Successfully uninstalled accelerate-1.5.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.6/474.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.1/219.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## Impor Libraries dan mengecek device yang digunakan

In [2]:
import sys
import torch
import transformers
import datasets
import evaluate
import accelerate

# Mendeteksi GPU (CUDA) secara otomatis
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Semua library berhasil diimpor.")

2025-07-20 13:12:52.932874: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753017173.157808      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753017173.224226      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda
Semua library berhasil diimpor.


## Impor lanjutan dan pengaturan environtment

In [3]:
import os
import json
import torch
from tqdm import tqdm
from transformers import BertTokenizer, EncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate


# Pengaturan environment variable (opsional, tapi baik untuk reproduktifitas)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## Muat tokenizer dan model

In [4]:
tokenizer = BertTokenizer.from_pretrained("cahya/bert2bert-indonesian-summarization")
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = EncoderDecoderModel.from_pretrained("cahya/bert2bert-indonesian-summarization")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/999M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Fungsi load_dataset_from_folder dan muat data

In [5]:
import json
from tqdm import tqdm
from datasets import Dataset

def load_dataset_from_folder(folder_path, limit=None):
    # (Fungsi ini tetap sama, tidak perlu diubah)
    data = []
    files = sorted(os.listdir(folder_path))
    if limit:
        files = files[:limit]
        
    for file_name in tqdm(files, desc=f"Loading from {os.path.basename(folder_path)}"):
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
            item = json.load(f)
            input_text = " ".join([" ".join(sent) for sent in item["clean_article"]])
            summary_text = " ".join([" ".join(sent) for sent in item["clean_summary"]])
            data.append({
                "article": input_text,
                "summary": summary_text
            })
    return data

# --- PERUBAHAN UTAMA DI SINI ---
# Path sekarang langsung menunjuk ke folder 'dev' di dalam dataset input Anda.
# Asumsi folder 'dev' ada di dalam 'xtreme'.
data_path = "/kaggle/input/liputan6-dataset/liputan6_data/xtreme/dev"

# Muat data training dan testing dari path yang sudah benar
train_data = load_dataset_from_folder(data_path, limit=1600)
test_data = load_dataset_from_folder(data_path, limit=400) # Menggunakan sumber yang sama

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Jumlah data training: {len(train_dataset)}")
print(f"Jumlah data testing: {len(test_dataset)}")

Loading from dev: 100%|██████████| 400/400 [00:00<00:00, 1679.50it/s]

Jumlah data training: 1600
Jumlah data testing: 400


Fungsi `preprocess_function` dan tokenisasi dataset

In [6]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = tokenizer(examples["article"], max_length=max_input_length, padding="max_length", truncation=True)
    targets = tokenizer(examples["summary"], max_length=max_target_length, padding="max_length", truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=["article", "summary"])
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=["article", "summary"])

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## Pengaturan metrik evaluasi (ROUGE)

In [7]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Ganti -100 dengan token padding karena label yang di-abaikan akan diisi dengan -100
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {key: value * 100 for key, value in result.items()}

## Pengaturan argumen training dan `Seq2SeqTrainer`

In [8]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch # Pastikan torch sudah diimport jika fp16=torch.cuda.is_available() digunakan

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2, # <-- TAMBAHKAN ATAU SESUAIKAN BARIS INI
    logging_dir='/kaggle/working/logs',
    logging_steps=100,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Memulai training
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:645: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train enc

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.589400,0.386004,40.852731,21.602951,34.297372,34.370470
2,0.408900,0.234790,48.208546,30.192879,41.734402,41.847676
3,0.282300,0.148736,57.917885,42.099724,52.337331,52.406175
4,0.195600,0.104935,67.587379,55.749465,63.828242,63.779158
5,0.152100,0.089872,71.254667,61.295621,68.299813,68.243259


/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...

TrainOutput(global_step=2000, training_loss=0.37221684741973876, metrics={'train_runtime': 1646.3428, 'train_samples_per_second': 4.859, 'train_steps_per_second': 1.215, 'total_flos': 4907669127168000.0, 'train_loss': 0.37221684741973876, 'epoch': 5.0})

## Evaluasi dan simpan model

In [9]:
# Evaluasi model
results = trainer.evaluate()
print(results)

# Simpan model dan tokenizer
output_model_dir = "/kaggle/working/indo_summary_model"
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"Model dan tokenizer berhasil disimpan di: {output_model_dir}")

/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


{'eval_loss': 0.08987226337194443, 'eval_rouge1': 71.2546665656851, 'eval_rouge2': 61.295621081801535, 'eval_rougeL': 68.29981294590331, 'eval_rougeLsum': 68.2432591522711, 'eval_runtime': 164.478, 'eval_samples_per_second': 2.432, 'eval_steps_per_second': 0.608, 'epoch': 5.0}
Model dan tokenizer berhasil disimpan di: /kaggle/working/indo_summary_model


## Inferensi/prediksi dengan artikel baru

In [10]:
import torch
from transformers import BertTokenizer, EncoderDecoderModel

# 1. Muat model dan tokenizer yang sudah disimpan
#    Ini adalah praktik terbaik untuk memastikan Anda menggunakan model final hasil training.
model_path = "/kaggle/working/indo_summary_model"
model = EncoderDecoderModel.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Pindahkan model ke perangkat yang benar (GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 2. Siapkan artikel berita yang ingin diringkas
#    Ganti teks di bawah ini dengan artikel berita apa pun yang ingin Anda coba.
artikel_berita = """
Kementerian Komunikasi dan Informatika (Kominfo) menegaskan kembali komitmennya untuk memberantas judi online yang semakin meresahkan masyarakat. Direktur Jenderal Aplikasi Informatika, Semuel Abrijani Pangerapan, menyatakan bahwa pihaknya telah memblokir lebih dari 2,5 juta konten judi online sepanjang tahun ini. "Kami tidak akan berhenti sampai di sini. Kami akan terus bekerja sama dengan penegak hukum dan penyedia platform untuk menutup semua akses ke konten negatif ini," ujar Semuel dalam konferensi pers di Jakarta, Senin. Ia juga mengimbau masyarakat untuk proaktif melaporkan situs atau aplikasi judi online melalui kanal aduan resmi Kominfo.
"""

# 3. Lakukan tokenisasi dan generate ringkasan
#    Ubah teks menjadi format yang dimengerti model, lalu generate output.
inputs = tokenizer(artikel_berita, return_tensors="pt", max_length=512, truncation=True).to(device)

# Generate ringkasan dengan beberapa parameter untuk hasil yang lebih baik
summary_ids = model.generate(
    inputs['input_ids'], 
    num_beams=4,         # Menggunakan beam search untuk kualitas yang lebih baik
    max_length=100,      # Panjang maksimal ringkasan (dalam token)
    early_stopping=True
)

# 4. Decode hasil dan tampilkan
#    Ubah hasil dari token ID kembali menjadi teks yang bisa dibaca.
ringkasan = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Tampilkan hasilnya untuk perbandingan
print("="*50)
print("ARTIKEL ASLI:")
print("="*50)
print(artikel_berita)
print("\n" + "="*50)
print("HASIL RINGKASAN:")
print("="*50)
print(ringkasan)

ARTIKEL ASLI:

Kementerian Komunikasi dan Informatika (Kominfo) menegaskan kembali komitmennya untuk memberantas judi online yang semakin meresahkan masyarakat. Direktur Jenderal Aplikasi Informatika, Semuel Abrijani Pangerapan, menyatakan bahwa pihaknya telah memblokir lebih dari 2,5 juta konten judi online sepanjang tahun ini. "Kami tidak akan berhenti sampai di sini. Kami akan terus bekerja sama dengan penegak hukum dan penyedia platform untuk menutup semua akses ke konten negatif ini," ujar Semuel dalam konferensi pers di Jakarta, Senin. Ia juga mengimbau masyarakat untuk proaktif melaporkan situs atau aplikasi judi online melalui kanal aduan resmi Kominfo.


HASIL RINGKASAN:
kementerian komunikasi dan informatika ( kominfo ) berkomitmen untuk memberantas judi online. sebab, situs atau aplikasi judi online telah memblokir lebih dari 2, 5 juta konten judi.
